# TTA: MODEL PRE-TRAINING AND DIAGNOSTICS

## Set-up

In [ ]:
!git clone -b features/tta https://github.com/Type-Here/fdsml_federated_learning.git fdsml

In [ ]:
%cd fdsml

In [ ]:
!git pull

## Dependencies

In [ ]:
!pip install imagecorruptions thop "setuptools<81"

## Dataset

In [ ]:
!python datasets_prep/prepare_gtsrb.py --splits test

# TTA RUN


## Create Corruptions

In [ ]:
!python -m iot.gtsrb_c
!ls dataset/gtsrb_c | wc -l

## Self Check

In [ ]:
# 5. self_check - the accumulation, against the same data in one batch.
#    No CLI exists for this one, so it is four lines here.
import itertools, torch
from iot.bn_bank import self_check
from iot.source_model import build_model, image_folder_loader, load_checkpoint

CKPT = ("models/gtsrb_ResNet18_FedAvg_a0.5_c4_le1_seed42_20260829-185510.pkl")

checkpoint = load_checkpoint(CKPT)
model, manager = build_model(checkpoint['metadata'], checkpoint['weights'])
loader = image_folder_loader('dataset/gtsrb/train', manager.transform_pipeline,
                             batch_size=128, num_workers=2)
# a few hundred images is enough, and self_check concatenates them all
print(self_check(model, list(itertools.islice(loader, 4)), manager.device))

In [ ]:
%env CKPT=models/gtsrb_ResNet18_FedAvg_a0.5_c4_le1_seed42_20260829-185510.pkl

In [ ]:
# 6. The branch walk. Same branches as the full run - fallback, bootstrap batch,
#    unseen corruptions, four arms - in minutes. Its numbers are meaningless.
!python -m iot.stream_eval --checkpoint $CKPT --max-batches 2 --out results/tta_smoke


In [ ]:
# 7. Recalibration: ImageNet BatchNorm statistics -> GTSRB's. Written once and
#    kept, so the Source model is one file rather than something recomputed.
#    Writes <checkpoint>_bn.pkl beside the original.
!python -m iot.source_model --checkpoint $CKPT --data dataset/gtsrb/train

In [ ]:
# 8. The study.
!python -m iot.stream_eval --checkpoint ${CKPT%.pkl}_bn.pkl --gtsrb-c dataset/gtsrb_c --out results/tta --batch-size 128 --num-workers 4

## Results

Bring back the three small files - `results/tta/conditions.csv`,
`batches.csv`, `summary.json`. They are all the analysis needs.